# Jaffle Shop database AI-agent

#### This notebook show-cases an OpenAI based AI-agent that allows non-technical users to query the Jaffle Shop with business questions in natural language.

#### Prerequisites: 
#### - Install necessary packages listed in `requirements.txt`.
#### - Set the environment variable OPENAI_API_KEY. If you run this from VS code, this can also be done by adding a .env file with the key.

#### First some imports:

In [ ]:
from agent import DatabaseAgent
import duckdb
import argparse
import logging
from datetime import datetime
import matplotlib.pyplot as plt
import os
from pprint import pprint
from prompts import build_sql_prompt, build_presentation_prompt_df, build_presentation_prompt_short
from prompts import build_chart_prompt, CHART_SCHEMA
from plots import build_plot
import numpy as np
from utils import rename_columns_human
from eval import PLOT_TEST_CASES, PLOT_TEST_CASES2, TEST_CASES3



#### Now, open a connection to the business database and initiate the agent. For now, we don't want the agent to log intermediate results, so we define null logger.


In [ ]:
def log(message):
    pass

db_path = '../data/jaffle_shop.duckdb'
conn = duckdb.connect(db_path)
agent = DatabaseAgent(conn, log=log, temperature=0)

#### The full workflow is explained in the following, but the agent also runs it just calling `agent(question)`.

#### First, let's have a look at the database schema we are using by running `agent.get_schema_summary()`. Below, we can see that the schema has a few columns with categorical values, that the LLM needs to know the range of, such as customer.loyalty_tier and orders.status. 

#### To add these values to the LLM inputs, we run `agent.get_special_columns_content()`.

In [ ]:
schema_summary = agent.get_schema_summary()
pprint(schema_summary)
special_columns = [{"table":"customers", "col": "loyalty_tier"},
                   {"table":"orders", "col": "status"}]
pprint(agent.get_special_columns_content(special_columns))

#### Next, we give the agent a question, from which the agent builds a prompt:

In [ ]:
question =  "Who are our top 10 customers by total spend? Display the result in a bar chart showing total spend on the y-axis and customer name on the x-axis."
sql_prompt = build_sql_prompt(question, schema_summary, agent.special_columns)
print("Generated SQL Prompt:")
print(sql_prompt)

#### The prompt is then fed by the agent into an llm, which returns a SQL-statement. The SQL-statement is tested by the guardrails, which verify that the statement starts with "select" and do not contain any of the words "insert", "update", "delete", "drop", "alter". After checking the statement, it is fed to the database which returns some output.

In [ ]:
sql_response_dict = agent.sql_query(sql_prompt)
print(f"Query succesful: {str(sql_response_dict['success'])}\n")
if sql_response_dict['success']:
    print(f"Query string:\n{sql_response_dict['sql_query']}\n")
    print(f"DB output: \n{sql_response_dict['query_result']}\n")
else:
    error_text = f"Encountered an error of type {sql_response_dict.get('error type', 'UNKNOWN')}: {sql_response_dict.get('error text', 'No error text provided.')}"
    print(error_text)


#### Based on the output type, we decide whether we should present the results as a table or as a string. If there is only one row in the dataframe, we choose a text-presentation, and if there are more, we present the result as a table.


In [ ]:
#query_result = sql_response_dict["query_result"]
query_result = rename_columns_human(sql_response_dict["query_result"])
sql_query = sql_response_dict["sql_query"]
result_columns_and_DT_dict = {col: str(dtype) for col, dtype in zip(query_result.columns, query_result.dtypes)}
if len(query_result) == 1:
            presentation_type = "PRESENTATION: TEXT"
else: presentation_type = "PRESENTATION: TABLE"
print(f"Determined presentation type: {presentation_type}")
        

#### If the presentation is to be in a text format, the agent asks the LLM for a text with one prompt, and if it is to be presented as a table, the agent asks for another.

#### In the case that the presentation type is a table, the agent asks the LLM to specify which kind of plot is suitable.

In [ ]:
if presentation_type == "PRESENTATION: TEXT":
    to_text_prompt = build_presentation_prompt_short(question, agent.schema_summary, sql_query, query_result.head())
    print("Prompt used for presentation generation:", to_text_prompt)   

    text_output = agent.get_llm_response(to_text_prompt)
    print("Text Output:", text_output)   

elif presentation_type == "PRESENTATION: TABLE":
    df_presentation_prompt = build_presentation_prompt_df(question, agent.schema_summary, sql_query, query_result.head())
    print("Prompt used for presentation generation:", df_presentation_prompt, "\n")
    df_intro_text = agent.get_llm_response(df_presentation_prompt)
    text_output = f"{df_intro_text}\n\n{query_result.to_string(index=False)}"
    print("Table presentation output:")
    print(text_output, "\n")
    chart_prompt = build_chart_prompt(question=question, sql=sql_query, columns=result_columns_and_DT_dict, preview_rows=query_result.head())
    chart_response_dict = agent.get_llm_response_jsonschema(chart_prompt, json_schema=CHART_SCHEMA)
    print("Generated chart response dict:")
    pprint(chart_response_dict)
    fig, ax = build_plot(chart_response_dict, query_result, log=log)
    #display(fig)

#### The workflow is wrapped up in `agent(question)`, which returns a dict with the results:

In [ ]:
result_dict = agent(question)

In [ ]:
if result_dict["success"]:
    print("Final result:")
    print(result_dict["text"])
    print(result_dict['sql query'])
    fig, ax = result_dict.get("chart", [None, None])
    if fig is not None:
        display(fig)
else:
    print("Failed to get a result.")
    print(f"Error details: {result_dict.get('error type', 'UNKNOWN')}: {result_dict.get('error text', 'No error text provided.')}")

#### Now let's loop over all the test cases

In [ ]:
test_case_results = []
for test_case_result in PLOT_TEST_CASES:
    test_dict = test_case_result.copy()
    print(f"Running test case: {test_case_result['id']}")
    question = test_case_result["question"]
    response = agent(question)
    #print("Response:")
    #pprint(response)
    #print("\n\n")
    test_dict["response"] = response
    test_case_results.append(test_dict)


In [ ]:
for test_case_result in test_case_results:
    question = test_case_result["question"]
    response = test_case_result["response"]
    print("-"*50)
    print("test id:", test_case_result["id"])
    print(f"Question: {question}")
    
    if response["success"]:
        print(response["text"])
        if response["presentation_type"] == "TABLE" and (response.get("chart", None) is not None):
            print("Saving and displaying chart...")
            fig, ax = response["chart"]
            display(fig)
            #plt.show(fig)
            #plt.close()
        
    else: 
        error_text = f"Encountered an error of type {response.get('error type', 'UNKNOWN')}: {response.get('error text', 'No error text provided.')}"
        log(error_text)
        print(error_text)


In [ ]:
#conn.close()